# ResearchAgent v5.0 Backend Server

Run cells 1, 2, 3 in order.

| Cell | What it does |
|------|-------------|
| 1 | Clones latest code from GitHub + installs packages |
| 2 | Set your API keys |
| 3 | Starts the server with a public ngrok URL |

Frontend: `https://prashant30091997.github.io/research-agent/`

---

In [ ]:
# CELL 1: Clone latest code + install dependencies
import os
%cd /content
if os.path.exists('/content/research-agent'):
    !rm -rf /content/research-agent
    print('Removed old clone')

!git clone https://github.com/prashant30091997/research-agent.git
%cd /content/research-agent/backend
!pip install -q fastapi uvicorn==0.29.0 httpx python-dotenv pydantic
!pip install -q pyngrok --upgrade

# Verify key files exist
for f in ['main.py', 'ai_router.py', 'tools/search_pubmed.py', 'tools/paper_download.py', 'tools/drive_ops.py', 'tools/academic_write.py', 'tools/create_doc.py', 'tools/session_mgr.py']:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f'  {status} {f}')
print('\nReady')

In [ ]:
# CELL 2: Set your API keys
import os

# At least one AI provider required
os.environ['ANTHROPIC_API_KEY'] = ''   # sk-ant-api03-... (console.anthropic.com)
os.environ['GOOGLE_API_KEY'] = ''       # AIzaSy... (aistudio.google.com/apikey)

# Optional
os.environ['SCOPUS_API_KEY'] = ''       # dev.elsevier.com/apikey/manage
os.environ['DEFAULT_MODEL'] = 'gemini-2.5-flash'

# OR load from Google Drive JSON file (uncomment below)
# from google.colab import drive
# drive.mount('/content/drive')
# import json
# keys = json.load(open('/content/drive/MyDrive/research_agent_api_keys.json'))
# os.environ['ANTHROPIC_API_KEY'] = keys.get('anthropic', '')
# os.environ['GOOGLE_API_KEY'] = keys.get('google', '')
# os.environ['SCOPUS_API_KEY'] = keys.get('scopus', '')

print('API Keys:')
print(f"  Anthropic: {'SET' if os.environ.get('ANTHROPIC_API_KEY') else 'not set'}")
print(f"  Google:    {'SET' if os.environ.get('GOOGLE_API_KEY') else 'not set'}")
print(f"  Scopus:    {'SET' if os.environ.get('SCOPUS_API_KEY') else 'not set (optional)'}")
print(f"  Model:     {os.environ.get('DEFAULT_MODEL')}")
print()
print('PubMed search: FREE (no key needed)')
print('Paper download: FREE (PMC, Unpaywall, Europe PMC)')
print('Google Drive: uses frontend OAuth token')

In [ ]:
# CELL 3: Start server with public URL

NGROK_TOKEN = ''  # Get free token at https://ngrok.com/signup

# Setup ngrok
import os, subprocess
if NGROK_TOKEN:
    # Write config directly to avoid version issues
    ngrok_dir = os.path.expanduser('~/.config/ngrok')
    os.makedirs(ngrok_dir, exist_ok=True)
    with open(os.path.join(ngrok_dir, 'ngrok.yml'), 'w') as f:
        f.write(f'version: "3"\nauthtoken: {NGROK_TOKEN}\n')
    print('ngrok token configured')
else:
    print('WARNING: No ngrok token. Get one at https://ngrok.com/signup')

from pyngrok import ngrok
public_url = ngrok.connect(8000)

print(f'\n{"="*60}')
print(f'BACKEND URL: {public_url}')
print(f'{"="*60}')
print(f'\n1. Open: https://prashant30091997.github.io/research-agent/')
print(f'2. Click Settings (bottom of sidebar)')
print(f'3. Paste Backend URL: {public_url}')
print(f'4. Save Settings')
print(f'\nTest: {public_url}/api/health')
print(f'{"="*60}')
print(f'\nKeep this cell running. Re-run all cells if Colab disconnects.\n')

proc = subprocess.Popen(['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'])

import time
try:
    while True:
        time.sleep(30)
        if proc.poll() is not None:
            print('Server stopped. Check errors above and re-run.')
            break
except KeyboardInterrupt:
    print('Stopped by user')
    proc.terminate()

In [ ]:
# OPTIONAL: Test the server (run while Cell 3 is running)
import requests
try:
    r = requests.get(f'{public_url}/api/health', timeout=10)
    data = r.json()
    print('Server Health:')
    for k, v in data.items():
        print(f'  {k}: {v}')
except Exception as e:
    print(f'Not responding: {e}')

In [ ]:
# OPTIONAL: Pull latest code without full re-clone
# Then restart Cell 3
%cd /content/research-agent
!git pull
%cd /content/research-agent/backend
print('Updated. Restart Cell 3 to apply.')